# Task 4.1 CNN Classifier in PyTorch

This notebook trains a custom CNN from scratch on the `A,B,CNNS_with_Tim` dataset, evaluates on the held-out test set, visualizes results, and prints predictions for `vacation_photos`.

## 1. Imports and Setup

In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score

try:
    import google.colab  # type: ignore
    in_colab = True
except Exception:
    in_colab = False

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    dev = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    dev = torch.device("mps")
else:
    dev = torch.device("cpu")

runtime_name = "Google Colab" if in_colab else "Local"
print("runtime:", runtime_name)
print("device:", dev)

## 2. Dataset Inspection

In [ ]:
root = Path("/content/A,B,CNNS_with_Tim") if in_colab else Path("A,B,CNNS_with_Tim")
tr_p = root / "veggie_heap_training"
te_p = root / "veggie_heap_testing"

img_size = 128
batch_size = 64
epochs = 15
lr = 0.001
weight_decay = 1e-4
val_split = 0.15
patience = 4
num_workers = 0

print("dataset root:", root)

if not tr_p.exists() or not te_p.exists():
    if in_colab:
        raise FileNotFoundError(
            "Dataset not found. Upload the full folder `A,B,CNNS_with_Tim/` to `/content` in Colab so these paths exist: "
            "`/content/A,B,CNNS_with_Tim/veggie_heap_training` and `/content/A,B,CNNS_with_Tim/veggie_heap_testing`. "
            "Do not upload only the inner class folders."
        )
    raise FileNotFoundError(
        "Dataset not found. Expected `A,B,CNNS_with_Tim/veggie_heap_training` and `A,B,CNNS_with_Tim/veggie_heap_testing` next to this notebook."
    )

In [ ]:
ds_ref = datasets.ImageFolder(tr_p)
cls = ds_ref.classes
y_all = np.array(ds_ref.targets)
c = len(cls)

cnt_tr = {cls[i]: int(np.sum(y_all == i)) for i in range(c)}
ds_te_ref = datasets.ImageFolder(te_p)
y_te_all = np.array(ds_te_ref.targets)
cnt_te = {cls[i]: int(np.sum(y_te_all == i)) for i in range(c)}

print("classes:", c)
for k in cls:
    print(f"- {k}")

print("\ntraining counts:")
for k, v in cnt_tr.items():
    print(f"{k}: {v}")

print("\ntesting counts:")
for k, v in cnt_te.items():
    print(f"{k}: {v}")

## 3. Transforms and Loaders

In [ ]:
def make_split(y, val_split=0.15, seed=42):
    rg = np.random.default_rng(seed)
    tr_idx, va_idx = [], []
    for i in np.unique(y):
        idx = np.where(y == i)[0]
        rg.shuffle(idx)
        nv = max(1, int(round(len(idx) * val_split)))
        va_idx.extend(idx[:nv].tolist())
        tr_idx.extend(idx[nv:].tolist())
    return sorted(tr_idx), sorted(va_idx)


def calc_stats(ds, idx, batch_size=64):
    dl = DataLoader(Subset(ds, idx), batch_size=batch_size, shuffle=False, num_workers=0)
    s1 = torch.zeros(3)
    s2 = torch.zeros(3)
    n = 0
    for X, _ in dl:
        s1 += X.sum(dim=(0, 2, 3))
        s2 += (X * X).sum(dim=(0, 2, 3))
        n += X.size(0) * X.size(2) * X.size(3)
    mu = s1 / n
    sd = torch.sqrt((s2 / n) - (mu * mu)).clamp_min(1e-6)
    return mu.tolist(), sd.tolist()


tr_idx, va_idx = make_split(y_all, val_split=val_split, seed=seed)

tf_stat = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])
ds_stat = datasets.ImageFolder(tr_p, transform=tf_stat)
mu, sd = calc_stats(ds_stat, tr_idx, batch_size=batch_size)

tf_tr = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.12),
    transforms.ToTensor(),
    transforms.Normalize(mu, sd),
])

tf_ev = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mu, sd),
])

ds_tr_full = datasets.ImageFolder(tr_p, transform=tf_tr)
ds_va_full = datasets.ImageFolder(tr_p, transform=tf_ev)
ds_te = datasets.ImageFolder(te_p, transform=tf_ev)

ds_tr = Subset(ds_tr_full, tr_idx)
ds_va = Subset(ds_va_full, va_idx)

vac_id = cls.index("vacation_photos")
vac_idx = [i for i, (_, t) in enumerate(ds_te.samples) if t == vac_id]
ds_vac = Subset(ds_te, vac_idx)

dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=num_workers)
dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=num_workers)
dl_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=num_workers)
dl_vac = DataLoader(ds_vac, batch_size=batch_size, shuffle=False, num_workers=num_workers)

cnt = np.bincount(y_all[tr_idx], minlength=c).astype(np.float32)
w = cnt.sum() / (c * cnt)
w = torch.tensor(w, dtype=torch.float32, device=dev)

print("mean:", [round(x, 4) for x in mu])
print("std:", [round(x, 4) for x in sd])
print("train:", len(ds_tr))
print("val:", len(ds_va))
print("test:", len(ds_te))
print("vacation test images:", len(ds_vac))

## 4. Model Definition

In [ ]:
class TaskCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.g = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * (img_size // 8) * (img_size // 8), 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, X):
        X = self.f(X)
        X = self.g(X)
        return X


m = TaskCNN(c).to(dev)
loss_fn = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=weight_decay)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)

print(m)
print("output classes:", c)

## 5. Training Loop

In [ ]:
def train_epoch(m, dl, loss_fn, opt, dev):
    m.train()
    loss_sum, ok, n = 0.0, 0, 0
    for X, y in dl:
        X = X.to(dev)
        y = y.to(dev)
        opt.zero_grad()
        out = m(X)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        loss_sum += loss.item() * X.size(0)
        ok += (out.argmax(1) == y).sum().item()
        n += X.size(0)
    return loss_sum / n, ok / n


@torch.no_grad()
def eval_epoch(m, dl, loss_fn, dev):
    m.eval()
    loss_sum, ok, n = 0.0, 0, 0
    for X, y in dl:
        X = X.to(dev)
        y = y.to(dev)
        out = m(X)
        loss = loss_fn(out, y)
        loss_sum += loss.item() * X.size(0)
        ok += (out.argmax(1) == y).sum().item()
        n += X.size(0)
    return loss_sum / n, ok / n


def fit(m, dl_tr, dl_va, loss_fn, opt, sch, epochs=15, patience=4):
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best = None
    best_acc = -1.0
    best_loss = np.inf
    wait = 0
    best_ep = 0
    for ep_i in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(m, dl_tr, loss_fn, opt, dev)
        va_loss, va_acc = eval_epoch(m, dl_va, loss_fn, dev)
        sch.step(va_loss)

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(va_loss)
        hist["train_acc"].append(tr_acc)
        hist["val_acc"].append(va_acc)

        cur_lr = opt.param_groups[0]["lr"]
        print(f"ep {ep_i}/{epochs}  lr {cur_lr:.6f}  tr_loss {tr_loss:.4f}  tr_acc {tr_acc:.4f}  val_loss {va_loss:.4f}  val_acc {va_acc:.4f}")

        better = (va_acc > best_acc) or (np.isclose(va_acc, best_acc) and va_loss < best_loss)
        if better:
            best_acc = va_acc
            best_loss = va_loss
            best = deepcopy(m.state_dict())
            best_ep = ep_i
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"early stopping at epoch {ep_i}")
                break

    if best is not None:
        m.load_state_dict(best)
    return hist, best_ep, best_acc, best_loss

## 6. Validation and Model Selection

In [ ]:
hist, best_ep, best_acc, best_loss = fit(m, dl_tr, dl_va, loss_fn, opt, sch, epochs=epochs, patience=patience)
print("best epoch:", best_ep)
print("best val acc:", round(float(best_acc), 4))
print("best val loss:", round(float(best_loss), 4))

## 7. Final Evaluation

In [ ]:
@torch.no_grad()
def get_preds(m, dl, dev):
    m.eval()
    ys, ps, pr = [], [], []
    for X, y in dl:
        X = X.to(dev)
        out = m(X)
        prob = torch.softmax(out, dim=1).cpu().numpy()
        pred = np.argmax(prob, axis=1)
        ys.append(y.numpy())
        ps.append(pred)
        pr.append(prob)
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(pr)


y_true, y_pred, y_prob = get_preds(m, dl_te, dev)
te_loss, te_acc = eval_epoch(m, dl_te, loss_fn, dev)
cm = confusion_matrix(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")
try:
    auc_macro = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
except ValueError:
    auc_macro = float("nan")
rep = classification_report(y_true, y_pred, target_names=cls)

print("test loss:", round(float(te_loss), 4))
print("test acc:", round(float(te_acc), 4))
print("macro f1:", round(float(f1_macro), 4))
print("weighted f1:", round(float(f1_weighted), 4))
print("macro roc auc:", round(float(auc_macro), 4))
print("\nclassification report:\n")
print(rep)

## 8. Visualizations

In [ ]:
def denorm(X, mu, sd):
    mu_t = torch.tensor(mu).view(3, 1, 1)
    sd_t = torch.tensor(sd).view(3, 1, 1)
    return (X.cpu() * sd_t) + mu_t


plt.figure(figsize=(7, 4))
plt.plot(hist["train_loss"], marker="o", label="train loss")
plt.plot(hist["val_loss"], marker="s", label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epochs")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(hist["train_acc"], marker="o", label="train acc")
plt.plot(hist["val_acc"], marker="s", label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epochs")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=cls, yticklabels=cls)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

rg = np.random.default_rng(seed)
show_idx = rg.choice(len(ds_te), size=min(12, len(ds_te)), replace=False)
plt.figure(figsize=(14, 10))
for j, i in enumerate(show_idx, 1):
    X, y = ds_te[i]
    with torch.no_grad():
        out = m(X.unsqueeze(0).to(dev))
        p = int(out.argmax(1).cpu().item())
    img = denorm(X, mu, sd).permute(1, 2, 0).numpy().clip(0, 1)
    plt.subplot(3, 4, j)
    plt.imshow(img)
    plt.title(f"p: {cls[p]}\nt: {cls[y]}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Vacation Predictions

In [ ]:
if len(vac_idx) == 0:
    print("No vacation images found in the test set.")
else:
    _, vac_pred, _ = get_preds(m, dl_vac, dev)
    print("vacation predictions:")
    for k, p in zip(vac_idx, vac_pred):
        fp = Path(ds_te.samples[k][0]).name
        print(fp, "->", cls[p])